<style>
/* RTL for Hebrew prose; code, diagrams and identifiers stay LTR. */
[dir="rtl"] { text-align: right; }
[dir="ltr"] { text-align: left; }
[dir="rtl"] pre, [dir="rtl"] code { direction: ltr; text-align: left; unicode-bidi: embed; }
[dir="rtl"] table { direction: rtl; }
[dir="rtl"] th, [dir="rtl"] td { text-align: right; }
</style>

<div dir="rtl">

# Basic LLM Agents

היום תהפכו את ה-retrieval שבניתם בבוקר ל-tool, בתוך agent שתכתבו בעצמכם.

</div>

<div dir="ltr">

```text
question -> LLM -> tool call -> execute -> observation -> LLM -> ... -> answer
```

</div>

<div dir="rtl">

**כללים**

1. בלי frameworks: LangChain, LangGraph, LlamaIndex agents, CrewAI, AutoGen, OpenAI Agents SDK. tool calling נייטיב של הספק מותר. הלולאה נכתבת בידיים.
2. Opik דולק מהתא הראשון. לא פרק נפרד. ככה עובדים.
3. כשמשהו נכשל, קוראים את ה-trace לפני שנוגעים ב-prompt. מחפשים את הצעד השגוי **הראשון**, לא את המשפט האחרון.
4. בלי planning מפורש. זה מחר.

תאים מסומנים **PROVIDED** (לקרוא, להריץ, להמשיך), **TODO** (אתם כותבים), או 🔴 **PLACEHOLDER** (חייב מילוי לפני העברת התרגיל).

</div>

<div dir="rtl">

---

## 0. Fill in before teaching

המחברת היא שלד. חמישה דברים ריקים בכוונה, כי הם תלויים בסיפור, בספק ובמה שתיתנו לסטודנטים.
כל השאר (לולאת ה-agent, dispatch, evaluation, ניתוח כשלים) לא תלוי דומיין.

| # | Placeholder | איפה | מה צריך |
|---|---|---|---|
| 1 | Scenario | §1 | התרחיש התפעולי, בהמשך לתרגיל הבוקר |
| 2 | Corpus | §3.1 | 10–15 מסמכים |
| 3 | Provider adapter | §2.2 | שלוש פונקציות שמחברות ל-API של הספק |
| 4 | Capabilities | §4.1 | ה-API/client/פונקציות שהסטודנטים מקבלים ועוטפים |
| 5 | Dev set | §10.1 | 10–15 שאלות, שבע קטגוריות |

חפשו `PLACEHOLDER` במחברת.

**אילוץ תוכן.** מה שתיתנו ב-§4 חייב להחזיק עובדות שלא נמצאות באף מסמך, ולהפך. אם מקור אחד עונה על הכול, ה-agent לא צריך לבחור, והתרגיל מתמוטט ל-RAG עם צעדים מיותרים.

**שרשרת תלות.** מסמך מזכיר ישות לפי מזהה ← ה-capability מתרגם אותו ← קריאה נוספת משלימה. זה מה שהופך את §9 לאמיתי, וזה מה שאי אפשר לקבע מראש.

</div>

<div dir="rtl">

---

## 1. Mission

> 🔴 **PLACEHOLDER: כתבו כאן את התרחיש**
>
> בהמשך ישיר לתרגיל הבוקר: אותו ארגון, אותו ארכיון.
>
> **הביט הנרטיבי:** ה-retrieval של הבוקר עבד. ואז התחילו לשאול שאלות שהתשובה להן פשוט לא במסמכים, אלא במקור אחר. מענה דורש שני מקורות, בסדר שתלוי במה שהחזירה הקריאה הראשונה. את הסדר אי אפשר לקבע, כי השאלה הבאה תדרוש אחר.
>
> **המשימה:** לתת למודל tools, ושהוא יחליט באיזה מקור להשתמש, באיזה סדר, ומתי יש לו מספיק כדי לענות.

### 🔴 Example questions

| # | שאלה | מה נדרש |
|---|------|---------|
| 1 | *«להחליף: נענית מהמסמכים בלבד»* | מסמכים |
| 2 | *«להחליף: נענית מ-capability אחת»* | capability |
| 3 | *«להחליף: מסמך, ואז capability לפי מה שהוא החזיר»* | מסמכים ← capability |
| 4 | *«להחליף: שתי קריאות capability שונות»* | שתי קריאות |
| 5 | *«להחליף: ידע כללי או משימת ניסוח»* | ללא tool |

שורה 5 נשארת. agent שקורא ל-tool בכל שאלה הוא agent איטי ויקר. ההחלטה **לא** לפעול היא חלק מהעבודה.

</div>

<div dir="rtl">

---

## 2. Setup

§2.1 ו-§2.3 מסופקים. §2.2 הוא placeholder.

</div>

In [ ]:
# PROVIDED - תלויות. להריץ פעם אחת אם צריך.
# %pip install --quiet opik
# %pip install --quiet <your-model-provider-sdk>
# %pip install --quiet <any client library your section-4 capabilities need>

In [ ]:
# PROVIDED - imports ו-Opik.
import json
import os
import time
from dataclasses import dataclass, field
from typing import Any, Callable, Optional

os.environ.setdefault("OPIK_PROJECT_NAME", "basic-agents-exercise")

# שני שמות שכל המחברת משתמשת בהם:
#   @track(name=...)   פותח span
#   update_trace(...)  מצמיד שם / tags / metadata ל-trace הנוכחי
# אם Opik לא זמין הם הופכים ל-no-op והמחברת עדיין רצה - אבל כל התרגיל בנוי
# על קריאת traces. לתקן לפני שממשיכים.
OPIK_ENABLED = False
try:
    import opik
    from opik import opik_context

    opik.configure(use_local=os.environ.get("OPIK_USE_LOCAL", "false").lower() == "true")
    OPIK_ENABLED = True
except Exception as exc:  # noqa: BLE001
    print("!! Opik לא פעיל (" + type(exc).__name__ + ": " + str(exc) + ")")

if OPIK_ENABLED:
    track = opik.track

    def update_trace(**kwargs):
        try:
            opik_context.update_current_trace(**kwargs)
        except Exception:  # noqa: BLE001
            pass
else:
    def track(*args, **kwargs):
        if args and callable(args[0]):
            return args[0]
        return lambda fn: fn

    def update_trace(**kwargs):
        return None

print("Opik enabled:", OPIK_ENABLED)

<div dir="rtl">

### 2.1 The provider contract: PROVIDED

הלולאה לא צריכה לדעת מי הספק. שלוש פונקציות מכירות אותו; כל השאר עובד מול response מנורמל.

</div>

<div dir="ltr">

```text
         provider SDK
              |
   +----------+-----------+
   |  3 adapter functions |   <- the only provider-specific code (2.2)
   +----------+-----------+
              |
   LLMResponse / ToolCall     <- provider-neutral from here down
              |
   tools, dispatch, agent loop, evaluation
```

</div>

<div dir="rtl">

ה-dataclasses למטה הם הממשק שאתם מממשים מולו.

</div>

In [ ]:
# PROVIDED - חוזה ה-response המנורמל. לא לשנות.
@dataclass
class ToolCall:
    """קריאת tool אחת שהמודל ביקש."""
    id: str            # המזהה של הספק - נדרש כדי להחזיר את התוצאה הנכונה
    name: str          # שם ה-tool
    arguments: dict    # כבר מפוענח מ-JSON


@dataclass
class LLMResponse:
    """תור אחד של המודל, מנורמל בין ספקים."""
    text: str                    # הטקסט הגלוי ("" אם המודל רק קרא ל-tools)
    tool_calls: list             # list[ToolCall]; ריק = המודל סיים
    stop_reason: str             # stop reason של הספק, לדיבוג
    assistant_message: dict      # תור ה-assistant בפורמט הספק, להוספה as-is
    usage: dict = field(default_factory=dict)
    raw: Any = None              # האובייקט המקורי


@dataclass
class ToolResult:
    """קריאת tool שהורצה, בדרך חזרה למודל."""
    tool_call_id: str
    name: str
    content: str
    is_error: bool = False

<div dir="rtl">

### 2.2 🔴 PLACEHOLDER: provider adapter

ממשו את שלוש הפונקציות מול **tool calling נייטיב** של הספק. לא פרוטוקול טקסטואלי שאתם מפרסרים.

**`assistant_message` זה מה שחשוב.** תור ה-assistant בפורמט של הספק, מוחזר **כמו שהוא**. לא לבנות מחדש מ-`text`. שם יושבים tool-call ids ו-thinking blocks שהבקשה הבאה צריכה. בחלק מהמודלים זו שגיאת API מיידית, באחרים דגרדציה שקטה. הבאג הנפוץ ביותר בתרגיל.

</div>

<div dir="ltr">

```python
# --- OpenAI-style ------------------------------------------------------
msg = resp.choices[0].message
LLMResponse(
    text=msg.content or "",
    tool_calls=[ToolCall(id=tc.id, name=tc.function.name,
                         arguments=json.loads(tc.function.arguments or "{}"))
                for tc in (msg.tool_calls or [])],
    stop_reason=resp.choices[0].finish_reason,
    assistant_message=msg.model_dump(exclude_none=True),
)
# one message per result:
#   {"role": "tool", "tool_call_id": r.tool_call_id, "content": r.content}

# --- Anthropic-style ---------------------------------------------------
LLMResponse(
    text="".join(b.text for b in resp.content if b.type == "text"),
    tool_calls=[ToolCall(id=b.id, name=b.name, arguments=b.input)
                for b in resp.content if b.type == "tool_use"],
    stop_reason=resp.stop_reason,
    assistant_message={"role": "assistant", "content": resp.content},
)
# one message for all results:
#   {"role": "user", "content": [{"type": "tool_result",
#                                 "tool_use_id": r.tool_call_id,
#                                 "content": r.content, "is_error": r.is_error}, ...]}
```

</div>

<div dir="rtl">

שימו לב להבדל בשלב האחרון. פיצול תוצאות למספר הודעות אצל ספק שרוצה אחת הוא שגיאת API, ואצל אחרים מלמד את המודל להפסיק לקרוא ל-tools במקביל. לכן `make_tool_result_messages` מחזירה **רשימה**.

</div>

In [ ]:
# ==========================================================================
# 🔴 PLACEHOLDER - PROVIDER ADAPTER. לממש את שלוש הפונקציות.
# ==========================================================================

MODEL = "<<REPLACE: model id>>"
LLM_SETTINGS = {
    # 🔴 פרמטרים של הספק, במקום אחד, כדי שאפשר יהיה לכוונן ב-§11 בלי לגעת ב-adapter.
    # למשל "max_tokens": 8000, "temperature": 0, ...
}

# 🔴 בניית ה-client. אם לספק יש אינטגרציה ל-Opik, לעטוף כאן - כל קריאת API
# תהפוך ל-span אוטומטית. אחרת ה-@track על call_llm עדיין נותן span אחד לקריאה.
client = None  # <<REPLACE>>


def to_provider_tools(tools: list) -> Any:
    """🔴 להמיר את ה-tool specs הניטרליים מ-§5 לפורמט שהספק מצפה לו.

    קלט: [{"name": str, "description": str, "parameters": <JSON Schema>}]

    OpenAI-style:    [{"type": "function", "function": {...}}]
    Anthropic-style: [{"name":..., "description":..., "input_schema": <parameters>}]
    """
    raise NotImplementedError("to_provider_tools")


@track(name="llm_call")
def call_llm(messages: list, tools: Optional[list] = None,
             system: Optional[str] = None, **overrides) -> LLMResponse:
    """🔴 קריאה אחת למודל. מחזירה LLMResponse מנורמל.

    messages  : השיחה עד כה, בפורמט הספק
    tools     : tool specs ניטרליים (להעביר דרך to_provider_tools), או None
    system    : system prompt, או None
    overrides : דריסות נקודתיות מעל LLM_SETTINGS

    דרישות: tool calling נייטיב, כל השדות של LLMResponse מלאים,
    ו-assistant_message הוא תור ה-assistant בפורמט הספק, as-is.
    """
    raise NotImplementedError("call_llm")


def make_tool_result_messages(results: list) -> list:
    """🔴 להפוך list[ToolResult] להודעות שיתווספו ל-messages.

    מחזירה רשימה, כדי שספק שרוצה הודעה אחת לכל תוצאה וספק שרוצה הודעה אחת
    לכולן יתאימו שניהם.
    """
    raise NotImplementedError("make_tool_result_messages")


def make_user_message(text: str) -> dict:
    """🔴 בדרך כלל פשוט {"role": "user", "content": text}."""
    return {"role": "user", "content": text}

In [ ]:
# PROVIDED - תצוגה. תלוי רק ב-LLMResponse, לא בספק.
def show(response: LLMResponse) -> None:
    print("stop_reason:", response.stop_reason)
    if response.text.strip():
        print("  [text]", response.text.strip()[:600])
    for call in response.tool_calls:
        print("  [tool_call]", call.name, json.dumps(call.arguments, ensure_ascii=False))
    if response.usage:
        print("  tokens:", response.usage)


def print_trajectory(trace: dict) -> None:
    print("Q: " + trace["question"])
    print("stop_reason=" + str(trace["stop_reason"]) + "  steps=" + str(trace["steps"]))
    for call in trace["tool_calls"]:
        flag = " [ERROR]" if call["is_error"] else ""
        print("  step " + str(call["step"]) + ": " + call["name"]
              + "(" + json.dumps(call["arguments"], ensure_ascii=False) + ")" + flag)
        print("       -> " + str(call["observation"])[:200].replace("\n", " | "))
    print("ANSWER: " + str(trace["answer"])[:800])

<div dir="rtl">

### 2.3 First call, first trace

אחרי שה-adapter ממומש, הריצו. אחר כך **פתחו את ה-trace ב-Opik**. צריך לדעת איפה דברים נוחתים לפני שיש שם משהו מעניין.

</div>

In [ ]:
# PROVIDED - קריאה פשוטה בלי tools. למצוא אותה ב-Opik.
@track(name="warmup")
def warmup():
    update_trace(name="warmup", tags=["setup"])
    resp = call_llm(messages=[make_user_message("ענה בדיוק: tracing works.")],
                    system="You are a concise assistant.")
    return resp.text


print(warmup())

<div dir="rtl">

> **תרגיל 2.א:** ב-Opik, פתחו את ה-trace בשם `warmup`. איפה מופיע ה-system prompt ואיפה הפלט? חמש דקות עכשיו חוסכות שעה בהמשך.

</div>

<div dir="rtl">

---

## 3. RAG as a tool

tool הוא פונקציית Python שהמודל רשאי לבקש שתריצו. זה הכול. מה שמעניין הוא ש**כל מה שמסביב ל-tool הוא חלק מה-prompt**:

- **שם**: `search_documents` מול `search` מול `rag`
- **תיאור**: מתי לפנות אליו, ומתי לא
- **ארגומנטים**: רק `query`? גם `top_k`?
- **מה הוא מחזיר**: כמה קטעים, באיזה פורמט, עם איזה metadata

אם אלה שגויים, שום כיוונון של ה-system prompt לא יציל. זה המקום הראשון לבדוק כשה-agent בוחר מקור לא נכון.

</div>

<div dir="rtl">

### 3.1 🔴 PLACEHOLDER: corpus

התא למטה מגדיר את **המבנה** עם שלושה מסמכים מזויפים, כדי שהמחברת תרוץ. להחליף תוכן.

- 10–15 מסמכים מספיקים. ערבוב סוגים (אירועים, נהלים, סיכומים) נותן ל-retriever מה להבחין בין.
- לשמור על המפתחות `doc_id` / `title` / `date` / `text`, או לעדכן את `fallback_search` בהתאם.
- **המסמכים לא מכילים את העובדות שיושבות ב-§4.** רצוי שמסמך אחד יפנה במפורש למקור האחר. אז agent שעונה מהארכיון נראה שגוי ב-trace.
- לשתול **שרשרת תלות** אחת לפחות: מסמך מזכיר מזהה, ה-capability מתרגם אותו.

טעינה מקבצים גם בסדר, רק שהשם `DOCUMENTS` יישאר.

</div>

In [ ]:
# ==========================================================================
# 🔴 PLACEHOLDER - CORPUS. להחליף בקורפוס של תרגיל הבוקר.
# ==========================================================================

DOCUMENTS = [
    {
        "doc_id": "DOC-001",
        "title": "<<REPLACE: רשומת אירוע>>",
        "date": "2026-01-01",
        "text": ("<<REPLACE. מתאר משהו שקרה, ומזכיר ישות לפי מזהה - בלי הפרט "
                 "שהשאלה תבקש. זה מה שמאלץ קריאה שנייה ל-capability ב-§4.>>"),
    },
    {
        "doc_id": "DOC-002",
        "title": "<<REPLACE: נוהל או מדיניות>>",
        "date": "2026-01-02",
        "text": "<<REPLACE. כלל או סף. מקור טוב לשאלות שנענות מהמסמכים בלבד.>>",
    },
    {
        "doc_id": "DOC-003",
        "title": "<<REPLACE: סיכום או תדריך>>",
        "date": "2026-01-03",
        "text": ("<<REPLACE. רצוי שיאמר במפורש שהמקור האחר הוא הסמכותי לשדות "
                 "מסוימים, כדי שטעות בבחירת מקור תיראה ב-trace.>>"),
    },
]

print(str(len(DOCUMENTS)) + " documents  (PLACEHOLDER)")

In [ ]:
# PROVIDED - retriever גיבוי (TF-IDF, בלי תלויות). רק אם ה-pipeline של הבוקר לא זמין.
import math
import re
from collections import Counter


def _tokenize(text: str) -> list:
    # אותיות לטיניות, ספרות ועברית. אם הקורפוס בשפה אחרת - להרחיב כאן.
    return re.findall(r"[a-z0-9\u0590-\u05ff]+", text.lower())


def _build_index(documents: list):
    doc_tokens = [_tokenize(d["title"] + " " + d["text"]) for d in documents]
    df = Counter()
    for toks in doc_tokens:
        df.update(set(toks))
    n = len(documents)

    def idf(term):
        return math.log((n + 1) / (df.get(term, 0) + 1)) + 1.0

    def vectorize(tokens):
        tf = Counter(tokens)
        vec = {t: (1.0 + math.log(c)) * idf(t) for t, c in tf.items()}
        norm = math.sqrt(sum(v * v for v in vec.values())) or 1.0
        return {t: v / norm for t, v in vec.items()}

    return df, idf, vectorize, [vectorize(toks) for toks in doc_tokens]


_DF, _IDF, _VECTORIZE, _DOC_VECS = _build_index(DOCUMENTS)


def fallback_search(query: str, top_k: int = 3) -> list:
    qvec = _VECTORIZE([t for t in _tokenize(query) if t in _DF])
    scored = []
    for doc, dvec in zip(DOCUMENTS, _DOC_VECS):
        score = sum(w * dvec.get(t, 0.0) for t, w in qvec.items())
        if score > 0:
            scored.append((score, doc))
    scored.sort(key=lambda pair: -pair[0])
    return [dict(d, score=round(s, 4)) for s, d in scored[:top_k]]


print([h["doc_id"] for h in fallback_search("replace", top_k=3)])

<div dir="rtl">

### 3.2 Write the tool: TODO

**ה-tool מחזיר מחרוזת, והמחרוזת נכנסת ישירות ל-context של המודל.** מה שתשימו בה הוא מה שיש לו לחשוב עליו.

החלטות שאתם מקבלים עכשיו, בין אם שמתם לב ובין אם לא:

- כמה קטעים? 3 זה הרגל, לא חוק. נסו 1. נסו 6.
- `doc_id` בפנים? (רוצים ציטוט? כן.)
- score? (המודל עושה משהו עם `0.4127`?)
- לחתוך מסמכים ארוכים או להעביר שלמים?
- מה מחזירים כשאין תוצאות? מחרוזת ריקה היא מלכודת. המודל ימלא את השקט. לומר במפורש.

</div>

In [ ]:
# TODO - לחבר את ה-retrieval של הבוקר.
@track(name="tool:search_documents")
def search_documents(query: str, top_k: int = 3) -> str:
    """חיפוש בארכיון. מחזיר מחרוזת מפורמטת.

    TODO:
      1. לקרוא ל-pipeline של הבוקר (או ל-fallback_search) עם query.
      2. לפרמט את התוצאות למחרוזת אחת. לפחות doc_id ו-text.
      3. לטפל במפורש במקרה של אפס תוצאות.
    """
    raise NotImplementedError("search_documents")


# בדיקה - להסיר הערה אחרי המימוש.
# print(search_documents("<<שאילתה שאמורה לפגוע בקורפוס>>", top_k=2))

<div dir="rtl">

> **תרגיל 3.א:** הדפיסו את הפלט לשאילתה אחת ב-`top_k=1` וב-`top_k=6`. העריכו את עלות ה-tokens של כל אחד. משלמים אותה **בכל צעד של כל ריצה**. רשמו מה בחרתם ולמה. חוזרים לזה ב-§11.

</div>

<div dir="rtl">

---

## 4. Add more sources

הארכיון הוא מקור אחד. ה-agent צריך עוד אחד לפחות, שמחזיק עובדות שאין במסמכים, אחרת הוא אף פעם לא צריך **לבחור** מקור.

**מה המקור ואיך ניגשים אליו נתון לכם.** ייתכן שזה client פנימי שהקורס כבר משתמש בו, endpoint, handle למאגר, או פונקציה שמישהו אחר כתב. אתם לא בונים אותו. אתם **עוטפים אותו כ-tool**, וזו מיומנות אחרת.

ההבדל חשוב, כי יכולת שנבנתה למתכנתים כמעט אף פעם לא מתאימה למודל:

| נבנה למתכנת | מה שהמודל צריך |
|---|---|
| אובייקט מקונן / JSON גולמי | טקסט שטוח, קריאה אחת |
| זורק חריגה על מזהה שגוי | משפט שאומר שהמזהה לא נמצא |
| 12 פרמטרים, 3 חובה | מעט ארגומנטים, כולם אופציונליים |
| מחזיר 400 שורות | הבודדות שעונות על השאלה |
| שדה בשם `st_cd` | משהו שהתיאור יכול להסביר |

כל העמודה הימנית היא העבודה של העטיפה שלכם. שום דבר מזה הוא לא באג ב-API. הוא נבנה לקורא אחר.

</div>

<div dir="rtl">

### 4.1 🔴 PLACEHOLDER: the given capabilities

הכניסו כאן את מה שהסטודנטים מקבלים: imports, בניית client, credentials מהסביבה, או פונקציות כתובות. בכל צורה שזה מגיע.

**כל מה שהתא צריך לעשות הוא להשאיר שתיים-שלוש callables ב-scope**, כשברור מה כל אחת מחזירה. שני stubs מסופקים כדי שהמחברת תרוץ.

**שני כללים:**

1. **חייב להחזיק עובדות שאין באף מסמך**, ורצוי שמסמך יפנה אליו כמקור סמכותי לשדות האלה.
2. **חייב להיות קישור בין השניים.** מסמך מזכיר ישות לפי מזהה, ה-capability מתרגם אותו למה שהשאלה באמת ביקשה. זו השרשרת שהופכת את §9 לאמיתי.

אם הדבר האמיתי דורש credentials או רשת שאולי לא תהיה בכיתה, להשאיר גרסת stub מאחורי דגל, שאף אחד לא ייתקע.

</div>

In [ ]:
# ==========================================================================
# 🔴 PLACEHOLDER - היכולות שהסטודנטים מקבלים.
# להחליף ב-API client / קריאת שירות / פונקציה אמיתיים.
# ==========================================================================

# 🔴 כל setup שהיכולת האמיתית דורשת, למשל:
#     from acme_internal import RosterClient
#     roster = RosterClient(base_url=os.environ["ROSTER_URL"],
#                           token=os.environ["ROSTER_TOKEN"])
#
# credentials מהסביבה, ורצוי read-only: כל קריאה כאן היא קריאה בלבד,
# והארגומנטים נקבעים על ידי ה-agent.

USE_STUB_CAPABILITIES = True   # 🔴 לכבות אחרי חיבור אמיתי


def lookup_a(**criteria):
    """🔴 PLACEHOLDER - היכולת הראשונה, בדיוק כפי שהסטודנטים מקבלים אותה.

    לשנות שם למה שזה באמת. לתעד כאן את שני הדברים שצריך כדי לעטוף:
      - מה זה מחזיר   (רשימת רשומות? רשומה? אובייקט תגובה?)
      - איך זה נכשל   (מחזיר ריק? זורק? מחזיר שדה שגיאה?)

    ה-stub מחזיר list[dict] ו-[] כשאין התאמה. אם ה-API האמיתי זורק על miss
    או מחזיר מעטפת מקוננת - לכתוב את זה כאן, העטיפה ב-§4.2 צריכה להתמודד.
    """
    if not USE_STUB_CAPABILITIES:
        raise NotImplementedError("לחבר את היכולת האמיתית")
    records = [
        {"id": "A-1", "name": "<<REPLACE>>", "status": "<<REPLACE>>"},
        {"id": "A-2", "name": "<<REPLACE>>", "status": "<<REPLACE>>"},
    ]
    return [r for r in records
            if all(str(r.get(k, "")).lower() == str(v).lower()
                   for k, v in criteria.items() if v is not None)]


def lookup_b(**criteria):
    """🔴 PLACEHOLDER - היכולת השנייה. שימו לב ל-`a_id`: הקישור חזרה למקור
    הראשון הוא מה שמאפשר שאלות רב-שלביות."""
    if not USE_STUB_CAPABILITIES:
        raise NotImplementedError("לחבר את היכולת האמיתית")
    records = [
        {"id": "B-1", "a_id": "A-1", "name": "<<REPLACE>>"},
        {"id": "B-2", "a_id": "A-2", "name": "<<REPLACE>>"},
    ]
    return [r for r in records
            if all(str(r.get(k, "")).lower() == str(v).lower()
                   for k, v in criteria.items() if v is not None)]


print(lookup_a(id="A-1"))
print(lookup_b(a_id="A-1"))

<div dir="rtl">

### 4.2 Wrap them as tools: TODO

tool אחד לכל יכולת. אותן שלוש החלטות כמו ב-§3 (שם, ארגומנטים, מחרוזת מוחזרת), ועוד אלה שצצות רק כשעוטפים משהו שלא אתם כתבתם:

- **אילו ארגומנטים לחשוף?** לא את כולם. ארגומנט שהמודל לא יודע למלא נכון גרוע מארגומנט שאין. הוא מייצר קריאות בטוחות ושגויות. מעט, אופציונליים, בשמות שמתאימים לניסוח השאלה ולא למאגר.
- **כמה מהרשומה להחזיר?** בשאלה אחת חצי מהשדות רעש, בשאלה הבאה החצי שזרקתם הוא התשובה. פורמט קבוע לא יכול לשניהם. תחליטו, ותדעו להצדיק ב-§12.
- **איך נראית תוצאה ריקה?** `""` היא מלכודת. לומר שלא נמצא, ולומר מה חיפשתם.
- **איך נראה כשל?** אם היכולת זורקת או מחזירה שגיאה, העטיפה הופכת אותה למשפט שהמודל יכול לפעול לפיו. §7 תופס מה שבורח, אבל הודעה שנכתבה כאן שימושית הרבה יותר.

</div>

In [ ]:
# TODO - לעטוף כל יכולת כ-tool. לשנות שמות לפני שהסטודנטים רואים.
@track(name="tool:get_a")
def get_a(id: str = None, name: str = None, status: str = None) -> str:
    """מחזיר מחרוזת מפורמטת למודל.

    TODO:
      1. לקרוא ל-lookup_a עם הקריטריונים שהתקבלו.
      2. להפוך את התוצאה למחרוזת אחת. להחליט אילו שדות מצדיקים את מקומם.
      3. לטפל במפורש ב"לא נמצא".
      4. לטפל בכשל של היכולת - שלא תברח חריגה גולמית.
    """
    raise NotImplementedError("get_a")


@track(name="tool:get_b")
def get_b(id: str = None, a_id: str = None, name: str = None) -> str:
    """מחזיר מחרוזת מפורמטת למודל. אותם ארבעה שלבים, מול lookup_b."""
    raise NotImplementedError("get_b")


# בדיקות - להסיר הערה אחרי המימוש.
# print(get_a(id="A-1"))
# print(get_b(a_id="A-1"))

<div dir="rtl">

---

## 5. Tool schemas

הפונקציות קיימות, אבל המודל מעולם לא ראה אותן. מה שהוא רואה הוא **schema**: שם, תיאור בשפה טבעית, ו-JSON Schema לארגומנטים.

זו הפרוזה בעלת המנוף הגבוה ביותר במערכת. אין למודל מידע אחר על ה-tools. אם שניים נשמעים דומה, הוא יבחר את זה שהתיאור שלו נשמע רלוונטי יותר, לא בהכרח זה שהתכוונתם אליו.

- לומר **מתי להשתמש**, לא רק מה זה. *"חיפוש בארכיון"* חלש. *"חיפוש בדוחות, סיכומים ונהלים. להשתמש כדי לברר מה קרה באירוע או מה נוהל דורש"* חזק.
- לומר **מתי לא**, כשיש tool שכן שאפשר להתבלבל איתו.
- לתאר **כל ארגומנט**, עם הפורמט הצפוי.
- לשמור על גבולות חדים בין tools. חפיפה מייצרת הטלת מטבע.

ה-schemas נכתבים בצורה **ניטרלית** (`name` / `description` / `parameters`), ו-`to_provider_tools` מ-§2.2 ממיר.

</div>

In [ ]:
# TODO - למלא ולשפר את ה-schemas. הראשון כתוב כדוגמה, השניים האחרים stubs.
TOOLS = [
    {
        "name": "search_documents",
        "description": (
            "🔴 PLACEHOLDER - לשכתב לדומיין שלכם, לשמור על המבנה. "
            "חיפוש בארכיון: <<סוגי המסמכים>>. להשתמש כדי לברר <<אילו שאלות>>. "
            "הארכיון אינו מכיל <<השדות שיושבים ב-§4>> - לאלה יש tools אחרים."
        ),
        "parameters": {
            "type": "object",
            "properties": {
                "query": {"type": "string",
                          "description": "שאילתה בשפה טבעית, למשל '<<REPLACE>>'."},
                "top_k": {"type": "integer",
                          "description": "כמה מסמכים להחזיר. ברירת מחדל 3."},
            },
            "required": ["query"],
        },
    },
    {
        # TODO: שם טוב. תיאור שאומר מתי כן ומתי לא. תיאור לכל ארגומנט.
        "name": "get_a",
        "description": "Look up a record.",
        "parameters": {
            "type": "object",
            "properties": {
                "id": {"type": "string", "description": "TODO"},
                # TODO: אילו filters עוד שווים חשיפה?
                # ארגומנט שהמודל לא ידע למלא גרוע מארגומנט שאין.
            },
            "required": [],
        },
    },
    {
        # TODO: הכול.
        "name": "get_b",
        "description": "Look up a record.",
        "parameters": {
            "type": "object",
            "properties": {"name": {"type": "string", "description": "TODO"}},
            "required": [],
        },
    },
]

print([t["name"] for t in TOOLS])

In [ ]:
# PROVIDED - probe שמראה איזה tool המודל היה בוחר, בלי להריץ אותו.
@track(name="tool_selection_probe")
def probe_tool_choice(question: str, tools=None, system: str = None) -> dict:
    tools = TOOLS if tools is None else tools
    update_trace(name="probe", tags=["probe"], metadata={"question": question})
    resp = call_llm(messages=[make_user_message(question)], tools=tools,
                    system=system or "You are a helpful assistant.")
    return {"question": question,
            "calls": [{"name": c.name, "arguments": c.arguments} for c in resp.tool_calls],
            "text": resp.text[:200]}


# 🔴 PLACEHOLDER - ארבע שאלות מהדומיין, אחת לכל החלטה שרוצים לבחון.
PROBE_QUESTIONS = [
    "<<REPLACE: אמור לבחור בבירור capability>>",
    "<<REPLACE: אמור לבחור בבירור חיפוש מסמכים>>",
    "<<REPLACE: דו-משמעי - המעניינת>>",
    "<<REPLACE: אמור לא לבחור tool בכלל>>",
]


def run_probes(tools=None, system=None):
    for q in PROBE_QUESTIONS:
        out = probe_tool_choice(q, tools=tools, system=system)
        picked = ", ".join(c["name"] + str(c["arguments"]) for c in out["calls"]) or "(no tool)"
        print("Q: " + q)
        print("   -> " + picked)

<div dir="rtl">

> **תרגיל 5.א. לא לדלג.**
>
> 1. הריצו `run_probes()` עם התיאורים הגולמיים. רשמו מה נבחר לכל שאלה.
> 2. פתחו את ארבעת ה-traces ב-Opik. הסתכלו על רשימת ה-tools שנשלחה בפועל.
> 3. שפרו את שני התיאורים החסרים. ספציפית: מתי כן ומתי לא.
> 4. הריצו שוב והשוו לשלב 1.
>
> השאלה: **אילו מילים בדיוק שינו את ההחלטה?** לא "השתפר", אילו מילים. זו המיומנות.

</div>

In [ ]:
run_probes()

<div dir="rtl">

---

## 6. One manual cycle

לפני שכותבים לולאה, מריצים סיבוב אחד ביד. חמישה מהלכים, תמיד אותם:

</div>

<div dir="rtl">

1. קריאה ל-LLM עם ה-tools
2. להסתכל מה הוא ביקש: שם וארגומנטים
3. להריץ את פונקציית ה-Python
4. להוסיף את התוצאה לשיחה
5. לקרוא ל-LLM שוב

</div>

<div dir="rtl">

שום דבר לא מוסתר. אין framework שעושה את שלב 4, ושם רוב הבאגים.

</div>

In [ ]:
# TODO - סיבוב אחד ביד.
# 🔴 PLACEHOLDER: שאלה שדורשת קודם מסמך ואז capability לפי מה שהוא החזיר.
SCRATCH_QUESTION = "<<REPLACE: שאלה דו-מקורית>>"

messages = [make_user_message(SCRATCH_QUESTION)]

# --- שלב 1: קריאה ראשונה ---------------------------------------------------
first = call_llm(messages=messages, tools=TOOLS, system="You are a helpful assistant.")
show(first)

# --- שלב 2: לבחון את הבקשה --------------------------------------------------
# TODO: לשלוף את ה-ToolCall הראשון מ-first.tool_calls. מה ה-.name, .arguments, .id?
call = ...              # TODO
print("name:", ...)     # TODO
print("arguments:", ...)  # TODO
print("id:", ...)       # TODO

In [ ]:
# --- שלב 3: להריץ את הפונקציה -----------------------------------------------
# TODO: dispatch ידני הפעם - אם השם הוא "search_documents", לקרוא
#       search_documents(**call.arguments), וכן הלאה.
observation = ...       # TODO  (מחרוזת)
print(observation[:800])

In [ ]:
# --- שלב 4: להוסיף גם את תור ה-assistant וגם את התוצאה ----------------------
# תור ה-assistant חוזר as-is. לא לבנות מחדש מ-first.text - זה מפיל
# tool-call ids ו-thinking blocks שהספק החזיר.
messages.append(first.assistant_message)

# TODO: לבנות ToolResult ולהוסיף את ההודעות שהוא מייצר.
#       result = ToolResult(tool_call_id=call.id, name=call.name,
#                           content=observation, is_error=False)
#       messages.extend(make_tool_result_messages([result]))
...                     # TODO

# --- שלב 5: קריאה שנייה -----------------------------------------------------
second = call_llm(messages=messages, tools=TOOLS, system="You are a helpful assistant.")
show(second)

<div dir="rtl">

> **תרגיל 6.א:** פתחו את המסלול ב-Opik: שתי קריאות LLM, span אחד של tool ביניהן.
>
> - הקריאה השנייה ענתה, או ביקשה **tool נוסף**? אם ביקשה, זו בדיוק התלות הרב-שלבית, וזו הסיבה שצריך לולאה ולא שתי קריאות מקובעות.
> - מה ההבדל הפיזי בין הקלט של קריאה 1 לקריאה 2? (תשובה: עוד הודעות. רשימת ההודעות הגדלה **היא** הזיכרון של ה-agent. אין שום דבר אחר.)

</div>

<div dir="rtl">

---

## 7. Tool dispatch

המודל מחזיר שם של tool **כמחרוזת**. צריך להפוך את זה לקריאת Python, ולשרוד את זה שהמודל טועה, כי הוא יטעה.

שלושה מצבי כשל. כל אחד צריך להחזיר **מחרוזת שהמודל יכול לקרוא ולהתאושש ממנה**, לא לזרוק:

| כשל | דוגמה | למה קורה |
|---|---|---|
| tool לא מוכר | `"search_docs"` | המודל זכר חצי שם |
| ארגומנטים לא תקינים | `get_b(unit="A-1")` | מילת מפתח שגויה |
| שגיאת הרצה | ה-API לא זמין, הקריאה זורקת | קלט רע או באג ב-tool |

חריגה שבורחת הורגת את הריצה. מחרוזת שגיאה ברורה מאפשרת ל-agent לנסות שוב בצעד הבא, וזו התנהגות אמיתית שתראו ב-traces.

</div>

In [ ]:
# PROVIDED - מיפוי שם -> פונקציה. לעדכן שמות לפי ה-tools שלכם.
TOOL_FUNCTIONS = {
    "search_documents": search_documents,
    "get_a": get_a,
    "get_b": get_b,
}

In [ ]:
# TODO - לממש dispatch.
@track(name="execute_tool")
def execute_tool(tool_name: str, arguments: dict) -> tuple:
    """מריץ קריאת tool אחת. מחזיר (observation_string, is_error).

    TODO:
      1. tool לא מוכר -> הודעה עם השם והשמות התקינים, is_error=True.
      2. ארגומנטים לא תקינים -> לתפוס TypeError, להחזיר מה שגוי ומה החתימה מקבלת.
      3. כל שאר -> לתפוס Exception, להחזיר סוג והודעה. כאן גם צצה תקלת API.
      4. הצלחה -> (result, False). לוודא שזו מחרוזת.
    """
    raise NotImplementedError("execute_tool")

In [ ]:
# PROVIDED - בדיקות ל-execute_tool. כל ארבע צריכות להתנהג בהיגיון.
def _test_execute_tool():
    cases = [
        ("happy path",   "get_a",        {"id": "A-1"}),
        ("unknown tool", "search_docs",  {"query": "anything"}),
        ("bad argument", "get_b",        {"nonexistent_kwarg": "x"}),
        ("empty result", "get_a",        {"id": "NO-SUCH-ID"}),
    ]
    for label, name, args in cases:
        out, is_err = execute_tool(name, args)
        print("--- " + label + "  (is_error=" + str(is_err) + ")")
        print("    " + str(out)[:220].replace("\n", " | "))
        assert isinstance(out, str), label + ": observation must be a string"
    print("\nכל הארבע החזירו מחרוזת בלי לזרוק.")


# _test_execute_tool()

<div dir="rtl">

---

## 8. The agent loop

העיקר. עד כאן היו חלקים; זו המכונה.

בלי פרטי ה-API, agent הוא שלוש שורות:

</div>

<div dir="ltr">

```text
while not done:
    action      = LLM(state)
    observation = environment(action)
    state      += observation
```

</div>

<div dir="rtl">

- **state** היא רשימת ההודעות. זה כל מה ש"זיכרון" אומר כאן.
- **action** היא קריאת tool (או טקסט, כשסיים).
- **environment** היא ה-`execute_tool` שלכם.
- **done** הוא "המודל לא החזיר קריאות tool". המודל מחליט מתי יש לו מספיק, וזה בדיוק מה שהופך את זה ל-agent ולא ל-pipeline.

לשמור על מימוש שנכנס למסך אחד. מעבר לזה כבר בונים framework, וזה מה שאנחנו נמנעים ממנו היום.

</div>

In [ ]:
# 🔴 PLACEHOLDER - system prompt בסיסי. לשכתב לדומיין.
# הסטודנטים מכווננים אותו ב-§11, אז לשמור אותו פשוט בכוונה - prompt חזק
# מדי לא משאיר להם מה לשפר.
SYSTEM_PROMPT = """You are an assistant for <<REPLACE: התפקיד והארגון>>.

יש לך tools לחיפוש בארכיון ול-<<REPLACE: המקורות האחרים>>. השתמש בהם כשהתשובה
תלויה ברשומות האלה. ענה ישירות, בלי tool, כששאלה היא ידע כללי או משימת ניסוח.

בסס כל טענה עובדתית על פלט של tool. אם המידע לא קיים באף מקור, אמור זאת במפורש
במקום לנחש."""

In [ ]:
# TODO - לממש את הלולאה.
@track(name="run_agent")
def run_agent(question: str, max_steps: int = 5, tools=None, system_prompt: str = None) -> dict:
    """מריץ את ה-agent עד תשובה או עד סוף הצעדים.

    מחזיר: {"question", "answer", "steps", "tool_calls", "stop_reason", "messages"}
    כאשר כל איבר ב-tool_calls הוא
      {"step", "name", "arguments", "observation", "is_error"}
    ה-evaluation ב-§10 תלוי במפתחות האלה. לשמור עליהם.
    """
    tools = TOOLS if tools is None else tools
    system_prompt = SYSTEM_PROMPT if system_prompt is None else system_prompt

    messages = [make_user_message(question)]
    trace = {"question": question, "answer": None, "steps": 0,
             "tool_calls": [], "stop_reason": None, "messages": messages}

    for step in range(max_steps):
        trace["steps"] = step + 1
        response = call_llm(messages=messages, tools=tools, system=system_prompt)

        # TODO: להוסיף את תור ה-assistant ל-messages.
        #       response.assistant_message as-is. לא לבנות מחדש מטקסט.

        # TODO: אם response.tool_calls ריק - זו התשובה הסופית.
        #       trace["answer"] = response.text, trace["stop_reason"] = "answered",
        #       ולהחזיר trace.

        # TODO: אחרת, להריץ את כל הקריאות לפי הסדר. לכל אחת: execute_tool,
        #       לרשום dict ב-trace["tool_calls"] עם
        #       step / name / arguments / observation / is_error,
        #       ולבנות ToolResult עם ה-.id של אותה קריאה.

        # TODO: להוסיף את התוצאות עם
        #       messages.extend(make_tool_result_messages(results))
        #       להעביר את כל התוצאות של הסיבוב בקריאה אחת - ה-adapter מחליט
        #       אם הספק רוצה הודעה אחת או כמה.

    # TODO: לטפל במקרה שנגמרו הצעדים.
    #       trace["stop_reason"] = "max_steps".
    return trace

<div dir="rtl">

---

## 9. Single-tool, then multi-step

מתחילים בשאלות שדורשות קריאה אחת. כשאלה עובדות, עוברים לאלה שבהן **הארגומנט לקריאה השנייה קיים רק בפלט של הראשונה**:

</div>

<div dir="ltr">

```text
search_documents("<<the event>>")
  -> document mentions A-1
get_a("A-1")
  -> record mentions B-1
get_b("B-1")
  -> answer
```

</div>

<div dir="rtl">

אי אפשר לכתוב את השרשרת מראש. המודל מגלה אותה. הלולאה רק ממשיכה להזין observations עד שהוא מפסיק לבקש.

אחרי כל ריצה: **לפתוח את ה-trace.** לשאול אם המסלול היה הגיוני, לא רק אם התשובה נכונה. תשובה נכונה ממסלול גרוע היא באג שעוד לא צף.

</div>

In [ ]:
# 🔴 PLACEHOLDER - חימום. tool אחד, צעד אחד.
for q in [
    "<<REPLACE: נענית מהמסמכים בלבד>>",
    "<<REPLACE: נענית מקריאת capability אחת>>",
]:
    print_trajectory(run_agent(q))
    print("=" * 78)

In [ ]:
# 🔴 PLACEHOLDER - רב-שלבי. הקריאה השנייה תלויה ב-observation הראשון.
for q in [
    "<<REPLACE: מסמך, ואז capability לפי התוצאה>>",
    "<<REPLACE: שרשרת נוספת, רצוי בעומק שלושה צעדים>>",
]:
    print_trajectory(run_agent(q))
    print("=" * 78)

<div dir="rtl">

> **תרגיל 9.א:** לשרשרת העמוקה ביותר, פתחו את ה-trace ורשמו את רצף הקריאות המדויק. כמה צעדים? אפשר היה בפחות? הייתה קריאה מיותרת (אותו tool, אותם ארגומנטים, פעמיים)?
>
> **תרגיל 9.ב:** הריצו שאלה שלא אמורה לדרוש tool. הוא קרא למשהו? אם כן, זו קריאה מיותרת ופגם אמיתי. לרשום, מתקנים ב-§11.

</div>

<div dir="rtl">

---

## 10. Evaluation

שאלה אחת בכל פעם לא אומרת כלום. צריך dev set ויותר ממספר אחד.

**למה דיוק לבדו לא מספיק.** agent שקורא לכל ה-tools בכל שאלה יקבל ציון דיוק טוב ויהיה בלתי שמיש: איטי, יקר, בלתי אפשרי לדיבוג. agent שעונה נכון ממסלול **שגוי** יישבר ברגע שהשאלה תשתנה קצת. לכן מודדים:

| מדד | מה הוא תופס |
|---|---|
| answer accuracy | האם התשובה נכונה |
| required-tool usage | האם פנה למקורות שהיה חייב |
| unnecessary tool calls | האם קורא למה שלא צריך |
| avg tool calls | עלות והשהיה |
| tool execution errors | schemas שבורים, ארגומנטים רעים, tools שבירים |

כל ריצת evaluation מצמידה את מזהה הדוגמה ל-trace ב-Opik. כשמספר זז, הולכים ישר למסלול שהזיז אותו.

</div>

<div dir="rtl">

### 10.1 🔴 PLACEHOLDER: dev set

10–15 דוגמאות, נגזרות מ-§1, §3 ו-§4. שורה אחת כתובה במלואה כתבנית.

**כיסוי נדרש**: דוגמה אחת לפחות לכל קטגוריה:

| קטגוריה | מטרה |
|---|---|
| `rag_only` | מסמכים בלבד |
| `structured_only` | קריאת capability אחת |
| `rag_plus_structured` | מסמך, ואז capability לפי התוצאה |
| `multi_structured` | שתי קריאות capability |
| `multi_step` | שלוש קריאות תלויות ומעלה |
| `no_tool` | ידע כללי או ניסוח. כל קריאת tool היא פגם |
| `unavailable` | התשובה לא קיימת בשום מקום; agent תקין אומר זאת ולא ממציא |

**שדות**

| שדה | משמעות |
|---|---|
| `required_tools` | חייבים להיקרא לפחות פעם אחת |
| `allowed_tools` | כל קריאה מחוץ לרשימה נספרת כמיותרת |
| `must_include` / `must_not_include` | בדיקות substring (case-insensitive) |
| `judge` | קריטריון ל-LLM judge, איפה ש-substring לא עובד |

`must_include` כשלתשובה נכונה יש token ספציפי: שם, מזהה, מספר. `judge` ל-`no_tool` ול-`unavailable`, שם הנכונות היא **צורה** ("נמנע מלהמציא מספר") ולא מילה מסוימת.

**החטאות מכוונות הן השורות הכי שוות שתכתבו.** דוגמה שהשאילתה המתבקשת מחזירה רק חצי ממה שצריך, או ששני tools נשמעים בה סבירים, היא מה שהופך את §12 מטקס לאבחון אמיתי. לכתוב שתיים-שלוש בכוונה, ולסמן בעותק המרצה אילו הן, שלא ייחשבו לשבורות.

</div>

In [ ]:
# ==========================================================================
# 🔴 PLACEHOLDER - DEV SET. להחליף כל שורה.
# ==========================================================================

DEV_SET = [
    # ---- תבנית מלאה ------------------------------------------------------
    {
        "id": "ev01",
        "category": "rag_plus_structured",
        "question": "<<REPLACE: מסמך ואז capability>>",
        "required_tools": ["search_documents", "get_a"],
        "allowed_tools": ["search_documents", "get_a", "get_b"],
        "must_include": ["<<REPLACE: token שחייב להופיע בתשובה>>"],
        "must_not_include": [],
    },
    # ---- stubs: אחד לכל קטגוריה נותרת ------------------------------------
    {"id": "ev02", "category": "rag_only",
     "question": "<<REPLACE>>",
     "required_tools": ["search_documents"], "allowed_tools": ["search_documents"],
     "must_include": ["<<REPLACE>>"]},

    {"id": "ev03", "category": "structured_only",
     "question": "<<REPLACE>>",
     "required_tools": ["get_a"], "allowed_tools": ["get_a"],
     "must_include": ["<<REPLACE>>"]},

    {"id": "ev04", "category": "multi_structured",
     "question": "<<REPLACE>>",
     "required_tools": ["get_a", "get_b"], "allowed_tools": ["get_a", "get_b"],
     "must_include": ["<<REPLACE>>"]},

    {"id": "ev05", "category": "multi_step",
     "question": "<<REPLACE: שלוש קריאות תלויות>>",
     "required_tools": ["search_documents", "get_a", "get_b"],
     "allowed_tools": ["search_documents", "get_a", "get_b"],
     "must_include": ["<<REPLACE>>"]},

    {"id": "ev06", "category": "no_tool",
     "question": "<<REPLACE: ידע כללי או ניסוח>>",
     "required_tools": [], "allowed_tools": [],
     "judge": "<<REPLACE: איך נראית תשובה נכונה. לא נדרשת שום קריאה.>>"},

    {"id": "ev07", "category": "unavailable",
     "question": "<<REPLACE: משהו שלא מתועד בשום מקור>>",
     "required_tools": [], "allowed_tools": ["search_documents", "get_a", "get_b"],
     "judge": "התשובה אומרת שהמידע לא זמין במקורות. אסור לה להמציא ערך."},

    # 🔴 להוסיף עוד 3-8. יעד: 10-15 סה\"כ.
]

print(str(len(DEV_SET)) + " dev examples  (PLACEHOLDER)")
for cat in sorted({e["category"] for e in DEV_SET}):
    print("  " + cat + ": " + str(sum(1 for e in DEV_SET if e["category"] == cat)))

In [ ]:
# PROVIDED - ניקוד. לא תלוי דומיין.
@track(name="llm_judge")
def llm_judge(question: str, answer: str, criterion: str) -> bool:
    prompt = (
        "You are grading one answer from an assistant.\n\n"
        "QUESTION:\n" + question + "\n\n"
        "ANSWER:\n" + str(answer) + "\n\n"
        "CRITERION:\n" + criterion + "\n\n"
        "Reply with exactly one word: PASS or FAIL."
    )
    resp = call_llm(messages=[make_user_message(prompt)],
                    system="You are a strict but fair grader.")
    return resp.text.strip().upper().startswith("PASS")


def grade_answer(example: dict, answer) -> tuple:
    """מחזיר (is_correct, reason)."""
    if not answer:
        return False, "empty answer"
    low = str(answer).lower()
    for needle in example.get("must_include", []):
        if needle.lower() not in low:
            return False, "missing '" + needle + "'"
    for needle in example.get("must_not_include", []):
        if needle.lower() in low:
            return False, "contains forbidden '" + needle + "'"
    if example.get("judge"):
        if not llm_judge(example["question"], answer, example["judge"]):
            return False, "judge: FAIL"
    return True, "ok"

In [ ]:
# PROVIDED - runner. כל דוגמה מקבלת את המזהה שלה על ה-trace ב-Opik.
@track(name="eval_example")
def evaluate_one(example: dict, **agent_kwargs) -> dict:
    update_trace(
        name="eval::" + example["id"],
        tags=["eval", example["id"], example["category"]],
        metadata={"example_id": example["id"], "category": example["category"],
                  "question": example["question"]},
    )
    started = time.time()
    trace = run_agent(example["question"], **agent_kwargs)
    elapsed = time.time() - started

    called = [c["name"] for c in trace["tool_calls"]]
    required = example.get("required_tools", [])
    allowed = set(example.get("allowed_tools", []))

    correct, reason = grade_answer(example, trace["answer"])
    result = {
        "id": example["id"],
        "category": example["category"],
        "correct": correct,
        "reason": reason,
        "required_tools_used": all(t in called for t in required),
        "missing_tools": [t for t in required if t not in called],
        "unnecessary_calls": [t for t in called if t not in allowed],
        "n_tool_calls": len(called),
        "n_tool_errors": sum(1 for c in trace["tool_calls"] if c["is_error"]),
        "steps": trace["steps"],
        "stop_reason": trace["stop_reason"],
        "seconds": round(elapsed, 1),
        "answer": trace["answer"],
        "trace": trace,
    }
    update_trace(metadata={"example_id": example["id"], "correct": correct,
                           "n_tool_calls": result["n_tool_calls"]})
    return result


def evaluate(dev_set=None, **agent_kwargs) -> list:
    dev_set = DEV_SET if dev_set is None else dev_set
    results = []
    for example in dev_set:
        try:
            results.append(evaluate_one(example, **agent_kwargs))
        except Exception as exc:  # noqa: BLE001
            print("!! " + example["id"] + " crashed: " + type(exc).__name__ + ": " + str(exc))
            results.append({"id": example["id"], "category": example["category"],
                            "correct": False, "reason": "crash: " + str(exc),
                            "required_tools_used": False, "missing_tools": [],
                            "unnecessary_calls": [], "n_tool_calls": 0,
                            "n_tool_errors": 1, "steps": 0, "stop_reason": "crash",
                            "seconds": 0.0, "answer": None, "trace": None})
    return results


def report(results: list) -> None:
    n = len(results)
    print("=" * 78)
    print("PER-EXAMPLE")
    print("=" * 78)
    for r in results:
        mark = "PASS" if r["correct"] else "FAIL"
        extra = []
        if r["missing_tools"]:
            extra.append("missing=" + ",".join(r["missing_tools"]))
        if r["unnecessary_calls"]:
            extra.append("extra=" + ",".join(r["unnecessary_calls"]))
        if r["n_tool_errors"]:
            extra.append("tool_errors=" + str(r["n_tool_errors"]))
        print(r["id"] + "  " + mark + "  " + r["category"].ljust(20)
              + " calls=" + str(r["n_tool_calls"]) + " steps=" + str(r["steps"])
              + " " + ("; ".join(extra) or "")
              + ("" if r["correct"] else "   <- " + r["reason"]))

    print("=" * 78)
    print("SUMMARY  (n=" + str(n) + ")")
    print("=" * 78)
    print("answer accuracy          : " + str(round(100 * sum(r["correct"] for r in results) / n, 1)) + "%")
    print("required-tool usage      : " + str(round(100 * sum(r["required_tools_used"] for r in results) / n, 1)) + "%")
    print("examples w/ extra calls  : " + str(sum(1 for r in results if r["unnecessary_calls"])))
    print("avg tool calls / question: " + str(round(sum(r["n_tool_calls"] for r in results) / n, 2)))
    print("total tool errors        : " + str(sum(r["n_tool_errors"] for r in results)))
    print("hit max_steps            : " + str(sum(1 for r in results if r["stop_reason"] == "max_steps")))
    print("avg seconds / question   : " + str(round(sum(r["seconds"] for r in results) / n, 1)))

    print("\nBY CATEGORY")
    for cat in sorted({r["category"] for r in results}):
        rows = [r for r in results if r["category"] == cat]
        acc = round(100 * sum(r["correct"] for r in rows) / len(rows))
        print("  " + cat.ljust(22) + str(acc) + "%  (" + str(len(rows)) + ")")

In [ ]:
baseline_results = evaluate()
report(baseline_results)

<div dir="rtl">

---

## 11. Optimization challenge

לשפר את ה-agent על ה-dev set. מותר לשנות:

- system prompt
- שמות ותיאורים של tools
- schemas של ארגומנטים
- מה ה-tools מחזירים ואיך זה מפורמט
- פרמטרי retrieval: `top_k`, chunking
- `max_steps`
- טיפול בשגיאות ב-`execute_tool`
- כל מה שב-`LLM_SETTINGS`

אסור היום: planning מפורש, task decomposer, scratchpad, או framework. זה מחר.

**סדר העבודה:**

</div>

<div dir="rtl">

1. להריץ evaluation
2. לאתר כשל
3. לפתוח את ה-trace ב-Opik
4. למצוא את ההחלטה השגויה הראשונה, לא את המשפט האחרון
5. להעלות השערה אחת
6. לשנות דבר אחד
7. להריץ שוב

</div>

<div dir="rtl">

שתי טעויות נפוצות:

1. **מתקנים את הצעד האחרון.** התשובה שגויה כי שישה צעדים קודם נשלף המסמך הלא נכון. ניסוח מחדש של התשובה לא יזיז כלום.
2. **משנים חמישה דברים ביחד.** אז הציון זז ואין מושג ממה. דבר אחד. להריץ. לרשום את המספר.

| # | הכשל | הצעד השגוי הראשון | השערה | השינוי | דיוק לפני ← אחרי |
|---|---|---|---|---|---|
| 1 | | | | | |
| 2 | | | | | |
| 3 | | | | | |

</div>

In [ ]:
# שטח עבודה. להגדיר מחדש SYSTEM_PROMPT / TOOLS / גוף ה-tools למעלה, ולהריץ שוב.
tuned_results = evaluate()
report(tuned_results)


def compare(before, after):
    b = {r["id"]: r for r in before}
    print("changed examples:")
    for r in after:
        was = b.get(r["id"])
        if was and was["correct"] != r["correct"]:
            arrow = "FAIL -> PASS" if r["correct"] else "PASS -> FAIL"
            print("  " + r["id"] + "  " + arrow + "   " + r["reason"])
    n = len(after)
    print("accuracy: " + str(round(100 * sum(x["correct"] for x in before) / len(before)))
          + "%  ->  " + str(round(100 * sum(x["correct"] for x in after) / n)) + "%")
    print("avg calls: " + str(round(sum(x["n_tool_calls"] for x in before) / len(before), 2))
          + "  ->  " + str(round(sum(x["n_tool_calls"] for x in after) / n, 2)))


compare(baseline_results, tuned_results)

<div dir="rtl">

---

## 12. Failure analysis

לבחור **לפחות שתי דוגמאות שנכשלו** ולעבוד אותן לעומק. העוזר למטה מדפיס מסלול מלא; לקרוא לצידו את ה-trace ב-Opik.

### Failure categories

| קטגוריה | איך זה נראה |
|---|---|
| wrong tool | פנה לארכיון כשהתשובה במקור האחר, או להפך |
| missing tool | היה צריך קריאה שנייה ולא עשה אותה |
| unnecessary tool | קרא למשהו שהשאלה לא דרשה |
| bad arguments | tool נכון, שאילתה/מזהה/פילטר שגויים |
| retrieval failure | tool נכון ושאילתה סבירה, חזרו מסמכים לא נכונים |
| tool execution failure | ה-tool זרק, או הקריאה החזירה שגיאה |
| observation interpretation | observation נכון, המודל קרא אותו לא נכון |
| reasoning failure | observations נכונים, מסקנה שגויה |
| premature final answer | עצר כשעוד חסר מידע |
| hallucination | טען משהו שלא הופיע באף observation |

### Template

למלא פעמיים.

**כשל 1, מזהה: `____`**

1. **איזה מסלול התרחש?** (קריאות לפי הסדר, עם ארגומנטים)
2. **מה הצעד השגוי הראשון?** (מספר צעד, לא "התשובה")
3. **איזה סוג כשל?** (מהקטגוריות למעלה)
4. **למה זה קרה?** (מה במערכת גרם: תיאור, schema, פורמט, prompt, פרמטרי retrieval)
5. **איזה שינוי עשוי לתקן?** (ספציפי. "לשפר את ה-prompt" היא לא תשובה.)

---

**כשל 2, מזהה: `____`**

1. **איזה מסלול התרחש?**
2. **מה הצעד השגוי הראשון?**
3. **איזה סוג כשל?**
4. **למה זה קרה?**
5. **איזה שינוי עשוי לתקן?**

</div>

In [ ]:
# PROVIDED - הדפסת מסלול מלא לדוגמה אחת.
def inspect_result(results: list, example_id: str) -> None:
    row = next((r for r in results if r["id"] == example_id), None)
    if row is None:
        print("no such example: " + example_id)
        return
    print("=" * 78)
    print(row["id"] + "  [" + row["category"] + "]  "
          + ("PASS" if row["correct"] else "FAIL: " + row["reason"]))
    print("=" * 78)
    if row["trace"] is None:
        print("(crashed - no trajectory)")
        return
    trace = row["trace"]
    print("Q: " + trace["question"] + "\n")
    for call in trace["tool_calls"]:
        flag = "  [ERROR]" if call["is_error"] else ""
        print("STEP " + str(call["step"]) + "  " + call["name"] + flag)
        print("  args: " + json.dumps(call["arguments"], ensure_ascii=False))
        print("  obs : " + str(call["observation"])[:900])
        print("")
    print("stop_reason: " + str(trace["stop_reason"]) + "   steps: " + str(trace["steps"]))
    print("\nANSWER:\n" + str(trace["answer"]))


# כל הכשלים, ואז לבחור שניים לניתוח.
for r in tuned_results:
    if not r["correct"]:
        inspect_result(tuned_results, r["id"])

<div dir="rtl">

---

## 13. Final reflection

תשובות קצרות.

**1. מה הופך את זה ל-agent ולא ל-RAG רגיל?**
שניהם שולפים ואז מייצרים. הצביעו על השורה בקוד שלכם שבה ההבדל יושב.

**2. מה קובע את ההתנהגות, מלבד המודל?**
לא שיניתם מודל אף פעם והתנהגות השתנתה הרבה. מנו מה כן שיניתם, ודרגו את שלושת הראשונים לפי כמה הזיזו את המדדים.

**3. למה דיוק לבדו לא מספיק?**
תארו agent שיקבל 100% דיוק ועדיין לא ראוי לייצור.

**4. איך traces עזרו בדיבוג?**
כשל אחד קונקרטי שלא הייתם מאבחנים מהתשובה הסופית בלבד.

**5. מה נעשה קשה כשנדרשות הרבה יותר קריאות?**
הלולאה שלכם מסתדרת עם 3–4. דמיינו משימה שדורשת 40, על פני חצי שעה, שבה טעות בקריאה 6 מתגלה רק בקריאה 30.

- איפה רשימת ההודעות מפסיקה להיות זיכרון שמיש?
- איך תדעו **איזה** מבין 40 הצעדים היה השגוי?
- המודל מחליט על הצעד הבא מכל ההיסטוריה. מה קורה להחלטה הזו כשההיסטוריה היא 100k tokens של פלט tools?
- מה הייתם רוצים שה-agent יעשה **לפני** צעד 1, שהוא לא עושה היום?

הנקודה האחרונה היא התרגיל של מחר: **planning מפורש**, ניהול זיכרון, ו-agents ארוכי טווח.

</div>